In [ ]:
import pandas as pd
import numpy as np
import re

In [ ]:
df = pd.read_csv('/home/patel-krish/anaconda3/BrainyBeam/womens_clothing/dataset/womens_clothing.csv')

print("Original Columns & Dtypes:")
for col in df.columns:
    print(f"  {col!r:28s} | dtype={str(df[col].dtype):8s} | nunique={df[col].nunique():4d}")

print(f"\nOriginal Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

In [ ]:
class DynamicSchemaParser:
    """
    Standardizes column names to snake_case and dynamically detects and converts
    data types (boolean, numeric, datetime, category, string) without introducing NaNs.
    """
    def __init__(
        self,
        category_max_unique=20,
        category_max_ratio=0.05,
        numeric_parse_threshold=0.95,
        datetime_parse_threshold=0.95,
        text_cols=None
    ):
        self.category_max_unique = category_max_unique
        self.category_max_ratio = category_max_ratio
        self.numeric_parse_threshold = numeric_parse_threshold
        self.datetime_parse_threshold = datetime_parse_threshold
        self.text_cols = set(text_cols or [])
        self.log = []

    def _record(self, column, action, before, after):
        self.log.append({
            'column': column,
            'action': action,
            'before': str(before),
            'after': str(after)
        })

    @staticmethod
    def _to_snake_case(name):
        name = str(name).strip()
        name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', name)
        name = re.sub(r'[^a-zA-Z0-9]+', '_', name)
        return re.sub(r'_+', '_', name).strip('_').lower()

    def standardize_column_names(self, df):
        df = df.copy()
        new_cols, seen = [], {}
        for col in df.columns:
            clean = self._to_snake_case(col)
            if clean in seen:
                seen[clean] += 1
                clean = f"{clean}_{seen[clean]}"
            else:
                seen[clean] = 0
            new_cols.append(clean)
            if clean != col:
                self._record(clean, 'RENAME', col, clean)
        df.columns = new_cols
        return df

    def _is_boolean_like(self, s):
        non_null = s.dropna()
        if non_null.empty:
            return False
        bool_sets = [
            {0, 1}, {0.0, 1.0}, {True, False},
            {'0', '1'}, {'true', 'false'}, {'yes', 'no'}, {'y', 'n'}
        ]
        normalized = {str(v).strip().lower() if isinstance(v, str) else v for v in non_null.unique()}
        return len(normalized) <= 2 and any(normalized.issubset(bs) for bs in bool_sets)

    def _convert_boolean(self, s):
        mapping = {
            'true': True, 'yes': True, 'y': True, '1': True, 1: True, 1.0: True, True: True,
            'false': False, 'no': False, 'n': False, '0': False, 0: False, 0.0: False, False: False
        }
        return s.map(
            lambda x: mapping[str(x).strip().lower() if isinstance(x, str) else x] if pd.notna(x) else pd.NA
        ).astype('boolean')

    def _downcast_numeric(self, s):
        if pd.api.types.is_integer_dtype(s) or (pd.api.types.is_float_dtype(s) and (s.dropna() % 1 == 0).all()):
            return s.astype('Int64') if s.isna().any() else pd.to_numeric(s, downcast='integer')
        return pd.to_numeric(s, downcast='float')

    def parse_dtypes(self, df):
        df = df.copy()
        n_rows = len(df)

        for col in df.columns:
            s = df[col]
            orig_dtype = s.dtype
            orig_nulls = int(s.isna().sum())

            if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
                s = s.map(lambda x: x.strip() if isinstance(x, str) else x)

            if self._is_boolean_like(s):
                converted = self._convert_boolean(s)
                if int(converted.isna().sum()) == orig_nulls:
                    df[col] = converted
                    self._record(col, 'DTYPE_CONVERT', orig_dtype, df[col].dtype)
                    continue

            if pd.api.types.is_numeric_dtype(s):
                df[col] = self._downcast_numeric(s)
                if str(df[col].dtype) != str(orig_dtype):
                    self._record(col, 'DTYPE_DOWNCAST', orig_dtype, df[col].dtype)
                continue

            if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s):
                non_null = s.dropna()
                if len(non_null) > 0:
                    cleaned_num = non_null.astype(str).str.replace(r'[\$,%\s]', '', regex=True)
                    parsed_num = pd.to_numeric(cleaned_num, errors='coerce')
                    if parsed_num.notna().mean() >= self.numeric_parse_threshold and parsed_num.notna().all():
                        full_cleaned = s.map(lambda x: re.sub(r'[\$,%\s]', '', str(x)) if pd.notna(x) else np.nan)
                        converted = self._downcast_numeric(pd.to_numeric(full_cleaned, errors='coerce'))
                        if int(converted.isna().sum()) == orig_nulls:
                            df[col] = converted
                            self._record(col, 'DTYPE_CONVERT', orig_dtype, df[col].dtype)
                            continue

                    parsed_dt = pd.to_datetime(non_null, errors='coerce', format='mixed')
                    if parsed_dt.notna().mean() >= self.datetime_parse_threshold and parsed_dt.notna().all():
                        converted = pd.to_datetime(s, errors='coerce', format='mixed')
                        if int(converted.isna().sum()) == orig_nulls:
                            df[col] = converted
                            self._record(col, 'DTYPE_CONVERT', orig_dtype, df[col].dtype)
                            continue

                nunique = s.nunique(dropna=True)
                if col not in self.text_cols and nunique <= self.category_max_unique and (nunique / max(n_rows, 1)) <= self.category_max_ratio:
                    df[col] = s.astype('category')
                else:
                    df[col] = s.astype('string')
                self._record(col, 'DTYPE_CONVERT', orig_dtype, df[col].dtype)

        return df

    def fit_transform(self, df):
        return self.parse_dtypes(self.standardize_column_names(df))

    def get_report(self):
        return pd.DataFrame(self.log)

In [ ]:
parser = DynamicSchemaParser(text_cols=['title', 'review_text'])
df_standardized = parser.fit_transform(df)

print("=== Schema Transformation Report ===")
print(parser.get_report().to_string(index=False))

orig_mem = df.memory_usage(deep=True).sum() / 1024
new_mem = df_standardized.memory_usage(deep=True).sum() / 1024
nulls_added = int(df_standardized.isna().sum().sum() - df.isna().sum().sum())

print(f"\nMemory Usage: {orig_mem:.2f} KB -> {new_mem:.2f} KB ({(1 - new_mem / orig_mem) * 100:.1f}% reduction)")
print(f"Nulls Introduced: {nulls_added}")

In [ ]:
df_dirty = pd.DataFrame({
    '  Product ID!! ': ['101', '102', '103'],
    'UnitPrice($)': ['$29.99', ' $45.50 ', '$12.00'],
    'IsRecommended?': ['Yes', 'No', 'yes'],
    'Review Date': ['2026-01-15', '2026/02/20', '2026-03-05'],
    'Category Name': ['Tops', 'Dresses', 'Tops']
})

test_parser = DynamicSchemaParser(category_max_unique=5, category_max_ratio=0.7)
df_dirty_clean = test_parser.fit_transform(df_dirty)

print(df_dirty_clean.dtypes)

In [ ]:
df_standardized.to_parquet('womens_clothing_standardized.parquet', index=False)
df_standardized.to_csv('womens_clothing_standardized.csv', index=False)
print("Exported standardized dataset to Parquet and CSV.")